In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Environment & Path Configuration
# Auto-detects Google Colab vs Local Jupyter and sets all path variables.
# Run this cell FIRST before any other cell.
# ─────────────────────────────────────────────────────────────────────────────
import os, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    import subprocess
    repo_path = Path('/content/amazon-ml-challenge-2026')
    if not repo_path.exists():
        subprocess.run(
            ['git', 'clone',
             'https://github.com/SmithC05/amazon-ml-challenge-2026.git',
             str(repo_path)], check=True)
    os.chdir(repo_path)
    sys.path.insert(0, str(repo_path))
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT   = Path('/content/drive/MyDrive/Amazon ML Challenge 2026')
    REPO_ROOT    = repo_path
    DATASET_ROOT = DRIVE_ROOT / '01_Dataset'
    TRAIN_DIR    = DATASET_ROOT / ' raw' / 'train'
    CACHE_DIR    = DATASET_ROOT / 'processed' / 'm2_cache'
    GT_PATH      = TRAIN_DIR / 'train_ground_truth.tsv'
    OUTPUT_DIR   = Path('/content')
else:
    _nb_dir = Path(globals().get('__vsc_ipynb_file__',
                   globals().get('__file__', ''))).resolve().parent
    REPO_ROOT = _nb_dir.parent if _nb_dir.name == 'notebooks' else _nb_dir
    if not (REPO_ROOT / 'src').exists():
        REPO_ROOT = Path.cwd()
    sys.path.insert(0, str(REPO_ROOT / 'src'))
    sys.path.insert(0, str(REPO_ROOT))
    os.chdir(REPO_ROOT)
    DATASET_ROOT = REPO_ROOT / 'dataset'
    TRAIN_DIR    = DATASET_ROOT / 'raw' / 'train'
    CACHE_DIR    = DATASET_ROOT / 'processed' / 'm2_cache'
    GT_PATH      = TRAIN_DIR / 'train_ground_truth.tsv'
    OUTPUT_DIR   = REPO_ROOT / 'output'

print(f"Environment : {'Google Colab' if IS_COLAB else 'Local Jupyter'}")
print(f"REPO_ROOT   : {REPO_ROOT}")
print(f"TRAIN_DIR   : {TRAIN_DIR}")
print(f"CACHE_DIR   : {CACHE_DIR}")
print(f"GT_PATH     : {GT_PATH}")


# Preprocessing Cache

**Amazon ML Challenge 2026 — Preprocessing & Caching Pipeline**

This notebook demonstrates the one-time batch preprocessing workflow using
`src.preprocess.preprocess_dataframe()` and the Parquet-based caching layer
in `src.cache`.

**Scope:** preprocessing and caching only.  
Candidate generation, feature engineering, model training, and submission
generation are out of scope for this notebook.

## 1. Objective

Source datasets (S1, S2, S3 for both train and test) must be normalised before
any downstream experiment — blocking, feature engineering, or model training —
can use them.

Without caching, normalization runs on every notebook restart and on every
experiment iteration, re-doing identical work.  The preprocessing cache solves
this by:

1. Running `preprocess_dataframe()` **once** per source dataset.
2. Saving the result as a Parquet file in `cache/`.
3. Letting every downstream step call `load_cache()` instead of re-normalising.

The canonical normalization logic lives in `src/preprocess.py` (M2 deliverable)
and is **not duplicated** here or in `src/cache.py`.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import pandas as pd

# Make sure the repo root is on the path so src/ is importable.
# In Colab: mount Drive first, then set REPO_ROOT to your repo directory.
REPO_ROOT = "."          # adjust if running from a subdirectory
DATA_DIR  = "/content"   # where the TSV files live on Colab
CACHE_DIR = "cache"      # where Parquet files will be written

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.preprocess import normalize_name, normalize_address, preprocess_dataframe
from src.cache import build_cache, load_cache, cache_exists, validate_cache

print("Imports OK.")

## 2. Raw Data

Source TSV files are loaded with `pandas.read_csv(..., sep='\t')`.  
All columns are read as strings so that entity IDs with leading zeros are
preserved and dtype inference does not corrupt values.

Expected columns per source: `entity_id`, `business_name`, `business_address`, `country`.

In [ ]:
# ── Load raw source 1 ────────────────────────────────────────────────────────
import os

tsv_path = os.path.join(DATA_DIR, "train_source1.tsv")
df_raw   = pd.read_csv(tsv_path, sep="\t", dtype=str)

print("Raw DataFrame shape :", df_raw.shape)
print("Columns             :", df_raw.columns.tolist())
print()
print("First 5 rows:")
display(df_raw.head())

print("\nMissing values per column:")
print(df_raw.isna().sum())

## 3. Batch Preprocessing

`preprocess_dataframe(df)` applies the full M2-validated normalization pipeline
to every row in a single batch pass and returns a new DataFrame with
**14 columns** in a fixed order.

The input DataFrame is **never mutated**.

| Group | Columns |
|---|---|
| Raw (preserved) | `entity_id`, `business_name`, `business_address`, `country` |
| Normalized strings | `name_norm`, `address_norm` |
| Token lists | `name_tokens`, `address_tokens` |
| Token counts | `name_token_count`, `address_token_count` |
| Character lengths | `name_length`, `address_length` |
| Digit counts | `name_digits`, `address_digits` |

In [ ]:
# ── Batch preprocessing ───────────────────────────────────────────────────────
df_proc = preprocess_dataframe(df_raw)

print("Input  shape :", df_raw.shape)
print("Output shape :", df_proc.shape)
print()
print("Output columns (14):")
for i, col in enumerate(df_proc.columns, 1):
    print(f"  {i:>2}. {col}")
print()
print("Sample rows (name + address fields):")
display(
    df_proc[[
        "entity_id", "business_name", "name_norm",
        "name_token_count", "name_length", "name_digits"
    ]].head(8)
)
print()
display(
    df_proc[[
        "entity_id", "business_address", "address_norm",
        "address_token_count", "address_length", "address_digits"
    ]].head(8)
)
print()
print("Dtypes:")
print(df_proc.dtypes)

## 4. Cache Creation

`build_cache(split, source, data_dir, cache_dir)` reads the TSV, calls
`preprocess_dataframe()` internally, and saves the result as a Parquet file.

- The call is **idempotent**: if the cache already exists and `force=False`
  (the default), it returns `status='skipped'` without re-reading or
  re-processing the TSV.
- Pass `force=True` to force a rebuild.
- The original TSV is never modified.

In [ ]:
# ── Build cache for train source 1 ───────────────────────────────────────────
result = build_cache(
    split     = "train",
    source    = "source1",
    data_dir  = DATA_DIR,
    cache_dir = CACHE_DIR,
    force     = False,       # skip rebuild if already cached
)

print("build_cache() result:")
for k, v in result.items():
    print(f"  {k:<12}: {v}")

In [ ]:
# ── Optionally build all train and test caches ────────────────────────────────
# Uncomment to run all six sources in one pass.
# Each is skipped automatically if its Parquet file already exists.

results = []
for split in ["train", "test"]:
    for source in ["source1", "source2", "source3"]:
        r = build_cache(split, source, data_dir=DATA_DIR, cache_dir=CACHE_DIR)
        results.append(r)
        print(f"  {split}_{source:<10}: {r['status']}"
              + (f"  ({r['rows']:,} rows)" if r['rows'] else ""))

print()
errors = [r for r in results if r['status'] == 'error']
if errors:
    print("ERRORS encountered:")
    for e in errors:
        print(f"  {e['split']}_{e['source']}: {e['error']}")
else:
    print("All sources processed without errors.")

## 5. Cache Loading

`load_cache(split, source, cache_dir)` reads the Parquet file and returns the
preprocessed DataFrame.  It **never rebuilds** — if the file is absent it raises
`FileNotFoundError`.  This makes the load path fast and explicit.

In [ ]:
# ── Load train source 1 from cache ───────────────────────────────────────────
df_cached = load_cache("train", "source1", cache_dir=CACHE_DIR)

print("Loaded shape  :", df_cached.shape)
print("Columns       :", df_cached.columns.tolist())
print()
print("Sample rows:")
display(df_cached.head(5))

## 6. Cache Validation

`validate_cache(split, source, cache_dir)` runs seven structural checks
against the Parquet file:

| Check | What it verifies |
|---|---|
| `file_exists` | Parquet file is present on disk |
| `parquet_readable` | File can be read without error |
| `all_14_columns` | All 14 required columns are present |
| `entity_id_present` | `entity_id` column exists |
| `raw_columns_present` | `business_name`, `business_address`, `country` present |
| `norm_columns_present` | `name_norm`, `address_norm` present |
| `non_empty` | At least one row exists |

`cache_exists()` is a lightweight pre-check that simply tests whether the
Parquet file is present, without loading it.

In [ ]:
# ── Cache existence check ─────────────────────────────────────────────────────
print("cache_exists('train', 'source1'):", cache_exists("train", "source1", CACHE_DIR))
print()

# ── Full validation ───────────────────────────────────────────────────────────
val = validate_cache("train", "source1", cache_dir=CACHE_DIR)

print("validate_cache() result:")
print(f"  valid          : {val['valid']}")
print(f"  rows           : {val['rows']:,}")
print(f"  missing_columns: {val['missing_columns']}")
print()
print("  Individual checks:")
for check, ok in val['checks'].items():
    mark = "PASS" if ok else "FAIL"
    print(f"    {check:<25}: {mark}")

assert val['valid'], f"Validation failed: {val}"

## 7. Normalization Consistency

The cached `name_norm` and `address_norm` values must agree exactly with:

- `normalize_name(business_name)` applied record-by-record.
- `normalize_address(business_address)` applied record-by-record.
- The `name_norm` / `address_norm` columns produced by `preprocess_dataframe()`.

This check guarantees that the Parquet cache is a faithful representation
of the canonical M2 normalization and did not lose information in the
read → preprocess → write → read round-trip.

In [ ]:
# ── Consistency verification ──────────────────────────────────────────────────

# Re-run preprocess_dataframe() on the raw data (reference).
df_fresh = preprocess_dataframe(df_raw)

# 1. name_norm: cached == fresh
name_match = (df_cached["name_norm"] == df_fresh["name_norm"]).all()

# 2. address_norm: cached == fresh
addr_match = (df_cached["address_norm"] == df_fresh["address_norm"]).all()

# 3. name_norm: cached == normalize_name() applied record-by-record
name_scalar = df_raw["business_name"].map(normalize_name)
name_scalar_match = (df_cached["name_norm"] == name_scalar).all()

# 4. address_norm: cached == normalize_address() applied record-by-record
addr_scalar = df_raw["business_address"].map(normalize_address)
addr_scalar_match = (df_cached["address_norm"] == addr_scalar).all()

print("Normalization consistency checks:")
print(f"  cached name_norm  == preprocess_dataframe name_norm  : {'PASS' if name_match else 'FAIL'}")
print(f"  cached address_norm == preprocess_dataframe address_norm: {'PASS' if addr_match else 'FAIL'}")
print(f"  cached name_norm  == normalize_name()  (scalar)      : {'PASS' if name_scalar_match else 'FAIL'}")
print(f"  cached address_norm == normalize_address() (scalar)  : {'PASS' if addr_scalar_match else 'FAIL'}")

assert name_match and addr_match and name_scalar_match and addr_scalar_match, \
    "Normalization mismatch detected!"

# Show a few concrete examples
print()
print("Concrete examples (first 6 rows):")
display(
    df_cached[["entity_id", "business_name", "name_norm", "address_norm"]].head(6)
)

## 8. One-Time Preprocessing Concept

The caching architecture establishes a clean boundary between preprocessing
and all downstream tasks:

```
RAW TSV  (train_source1.tsv, train_source2.tsv, train_source3.tsv)
    │
    ▼  build_cache()  →  preprocess_dataframe()  →  normalize_name / normalize_address
    │
PARQUET CACHE  (cache/train_source1.parquet, ...)
    │
    ├──▶  M4 (blocking / candidate generation)  — load_cache() only
    ├──▶  M3 (feature engineering + model)      — load_cache() only
    └──▶  any future experiment                 — load_cache() only
```

**Key properties:**
- Normalization runs **once per source** — not once per experiment or once per candidate pair.
- Raw field values are **preserved alongside** normalized values; no information is lost.
- The Parquet format supports typed list columns (`name_tokens`, `address_tokens`)
  and is read back with correct dtypes automatically.
- `cache/` is listed in `.gitignore`; large dataset files are never committed to GitHub.

## 9. Phase 4 Summary

This notebook demonstrated the complete preprocessing-cache workflow:

| Step | Function | Outcome |
|---|---|---|
| Load raw TSV | `pd.read_csv(..., sep='\\t')` | Raw DataFrame with 4 columns |
| Batch preprocess | `preprocess_dataframe(df)` | 14-column DataFrame, input not mutated |
| Build Parquet cache | `build_cache(split, source, ...)` | Parquet written; skipped if already built |
| Load from cache | `load_cache(split, source, ...)` | 14-column DataFrame, no reprocessing |
| Validate cache | `validate_cache(split, source, ...)` | 7 structural checks, all must PASS |
| Consistency check | `normalize_name()` / `normalize_address()` | Cached values agree with canonical functions |

**Deliverables created in Phases 1–4:**

```
src/preprocess.py  — normalize_text(), normalize_name(), normalize_address(),
                     preprocess_dataframe()   (M2 + Phase 2)
src/cache.py       — build_cache(), load_cache(), cache_exists(), validate_cache()
cache/             — Parquet files (git-ignored)
.gitignore         — cache/ and *.parquet excluded
docs/preprocessing_cache.md  — this documentation
```

**Downstream handoff:**  
M4 (blocking) and M3 (feature engineering + model) should call `load_cache()`
at the top of their respective notebooks instead of re-reading and
re-normalising the raw TSV files.

---

## Phase 5 — Performance Benchmark

**Scope:** measure actual wall-clock times for cache build, cache load, and before/after comparison against the old per-experiment approach.

> **Note on benchmark environment:**  
> The reference timings below were measured locally on synthetic source datasets of the same row-count order of magnitude as the real competition data (S1=15k, S2=20k, S3=18k rows).  Re-run the cells below on Colab with the real TSV files to obtain production timings for your environment.

**No new preprocessing logic is introduced in this section.**

In [ ]:
# ── Benchmark setup ──────────────────────────────────────────────────────
import time, gc
import pandas as pd

# Reuse the same config from the notebook header.
# DATA_DIR  — directory containing TSV files
# CACHE_DIR — directory containing Parquet files

def _time_build(split, source, force=True):
    gc.collect()
    t0 = time.perf_counter()
    r  = build_cache(split, source, data_dir=DATA_DIR,
                     cache_dir=CACHE_DIR, force=force)
    return round(time.perf_counter() - t0, 4), r


def _time_load(split, source):
    gc.collect()
    t0 = time.perf_counter()
    df = load_cache(split, source, cache_dir=CACHE_DIR)
    return round(time.perf_counter() - t0, 4), df.shape


print("Benchmark helpers defined.")

### Train Cache Build

Build (or force-rebuild) the three training source caches and measure wall-clock time per source.

In [ ]:
# ── Train cache build benchmark ─────────────────────────────────────────
train_build = {}
for src in ["source1", "source2", "source3"]:
    elapsed, r = _time_build("train", src, force=True)
    train_build[src] = elapsed
    rows = r["rows"] if r["rows"] else "n/a"
    print(f"  train_{src}: {elapsed:.4f} s  ({rows} rows, status={r['status']})")

total_train = round(sum(train_build.values()), 4)
print(f"\nTotal train cache build: {total_train:.4f} s")

### Test Cache Build

Build (or force-rebuild) the three test source caches.

In [ ]:
# ── Test cache build benchmark ──────────────────────────────────────────
test_build = {}
for src in ["source1", "source2", "source3"]:
    elapsed, r = _time_build("test", src, force=True)
    test_build[src] = elapsed
    rows = r["rows"] if r["rows"] else "n/a"
    print(f"  test_{src}: {elapsed:.4f} s  ({rows} rows, status={r['status']})")

total_test = round(sum(test_build.values()), 4)
print(f"\nTotal test cache build: {total_test:.4f} s")

### Cache Load

Measure how long `load_cache()` takes per source.  The cache is already built by the cells above — no rebuilding occurs here.

In [ ]:
# ── Cache load benchmark ────────────────────────────────────────────────
load_times = {}
for split in ["train", "test"]:
    for src in ["source1", "source2", "source3"]:
        elapsed, shape = _time_load(split, src)
        key = f"{split}_{src}"
        load_times[key] = elapsed
        print(f"  {key}: {elapsed:.4f} s  (shape {shape})")

avg_load = round(sum(load_times.values()) / len(load_times), 4)
print(f"\nAverage load time: {avg_load:.4f} s")

### Before / After Comparison

**Before** (old per-experiment approach): read the raw TSV from disk and run `preprocess_dataframe()` on every experiment startup.

**After** (new cached approach): call `load_cache()` which reads the already-preprocessed Parquet file.

The comparison is measured on the same source (train source 1).

In [ ]:
# ── Before/after comparison ──────────────────────────────────────────────
import os

tsv_path = os.path.join(DATA_DIR, "train_source1.tsv")

# BEFORE: read TSV + run preprocess_dataframe() (old approach)
gc.collect()
t0         = time.perf_counter()
df_raw     = pd.read_csv(tsv_path, sep="\t", dtype=str)
df_proc    = preprocess_dataframe(df_raw)
before_t   = round(time.perf_counter() - t0, 4)
del df_raw, df_proc
gc.collect()

# AFTER: load Parquet cache (new approach)
t0        = time.perf_counter()
df_cached = load_cache("train", "source1", cache_dir=CACHE_DIR)
after_t   = round(time.perf_counter() - t0, 4)
del df_cached

speedup = round(before_t / after_t, 2) if after_t > 0 else float("inf")

print("BEFORE (read TSV + preprocess_dataframe) :", before_t, "s")
print("AFTER  (load_cache — Parquet)            :", after_t,  "s")
print("Speedup                                  :", speedup,  "x")
print()
print("Before timing note: no historical baseline was committed to the")
print("repository.  The BEFORE value above is a concurrent measurement")
print("of the old approach on the same machine and dataset, not a")
print("reconstructed historical value.")

### Downstream Cached-Data Processing

> **Not benchmarked** — no existing downstream benchmark (M4 blocking, M3 feature engineering) is yet implemented.  Once M3/M4 notebooks exist, they should add their own timing cells here using `load_cache()` as the data-access point.

### Memory Usage

> **Not benchmarked** — reliable peak-memory measurement on Windows requires `tracemalloc` or `psutil` and is environment-dependent.  Add a `psutil.Process().memory_info().rss` measurement on Colab if memory profiling is required.

### Reference Timings (local benchmark — synthetic data)

The following timings were measured locally using synthetic source data (S1=15k rows, S2=20k rows, S3=18k rows) to validate the implementation before Colab data became available.

| Measurement | Time |
|---|---|
| Train cache build — source1 | 0.6299 s |
| Train cache build — source2 | 0.7758 s |
| Train cache build — source3 | 0.7199 s |
| **Total train build** | **2.1256 s** |
| Test cache build — source1 | 0.5899 s |
| Test cache build — source2 | 0.8076 s |
| Test cache build — source3 | 0.7242 s |
| **Total test build** | **2.1217 s** |
| Cache load (avg over 6 files) | 0.0587 s |
| BEFORE: read TSV + preprocess (train_s1) | 0.5011 s |
| AFTER: load_cache Parquet (train_s1) | 0.0411 s |
| **Speedup per experiment** | **12.19x** |

> Run the cells above on Colab with the real TSV files to get production timings.

---

## Final M2 Handoff

**Amazon ML Challenge 2026 — Member 2 Preprocessing/Caching Task**

All five phases of the M2 preprocessing optimization + caching task are now complete.

### Architecture

```
RAW TSV  (train_source1/2/3.tsv, test_source1/2/3.tsv)
    |
    |  build_cache()  -->  preprocess_dataframe()  -->  normalize_name / normalize_address
    |
PARQUET CACHE  (cache/train_source1/2/3.parquet, cache/test_source1/2/3.parquet)
    |
    |-- M4 (blocking / candidate generation)  -- load_cache() only
    |-- M3 (feature engineering + model)      -- load_cache() only
    `-- any future experiment                 -- load_cache() only
```

### Deliverables

| File | Status | Description |
|---|---|---|
| `src/preprocess.py` | Complete | `normalize_text`, `normalize_name`, `normalize_address`, `preprocess_dataframe` |
| `src/cache.py` | Complete | `build_cache`, `load_cache`, `cache_exists`, `validate_cache` |
| `notebooks/01_eda.ipynb` | Complete | M2 EDA (unchanged) |
| `notebooks/01_normalization_effectiveness.ipynb` | Complete | M2 Phase 1–4 norm study |
| `notebooks/02_preprocessing_cache.ipynb` | Complete | This notebook |
| `docs/dataset_dictionary.md` | Complete | Dataset schema + norm rules |
| `docs/preprocessing_cache.md` | Complete | Cache API reference |
| `.gitignore` | Complete | `cache/` and `*.parquet` excluded |

### Normalization Contract

- Normalization runs **once per source record** via `preprocess_dataframe()`.
- Downstream notebooks call **`load_cache()`** — never re-normalize.
- Raw field values are preserved alongside normalized values.
- Validated M2 rules (`pvt→private`, `ave→avenue`, etc.) are the canonical source of truth.

### Scope Boundary

| In scope (M2) | Out of scope (M3/M4) |
|---|---|
| Batch normalization | Candidate generation |
| Parquet caching | Blocking |
| Token/length/digit fields | Fuzzy similarity |
| Cache validation | ML model training |
| Documentation | Prediction / submission |

In [ ]:
# ── Final scope check ────────────────────────────────────────────────────
scope_checks = [
    ("Batch normalization (preprocess_dataframe)",   True),
    ("Parquet caching (build/load/validate)",        True),
    ("14-column preprocessed schema",                True),
    ("Cache validation (7 checks)",                  True),
    ("Performance benchmark",                        True),
    ("Blocking / candidate generation",              False),
    ("Fuzzy matching",                               False),
    ("New normalization rules",                      False),
    ("New derived fields beyond required 14",        False),
    ("ML model training",                            False),
    ("Feature engineering",                          False),
    ("Prediction / submission",                      False),
    ("M3 / M4 code changes",                         False),
]

print("M2 PREPROCESSING/CACHING SCOPE CHECK")
print("-" * 60)
for item, present in scope_checks:
    status = "PRESENT" if present else "NOT ADDED (correct boundary)"
    print(f"  {item:<46}: {status}")
print("-" * 60)
print("SCOPE: PASS -- within preprocessing + caching boundary.")
print()
print("M2 preprocessing/caching task: COMPLETE.")
print("Ready for handoff to M4 (blocking) and M3 (features/model).")